# Notebook 2: EDA & Playstyle Clustering
## Clustering Premier League Playstyles & Predicting Match Outcomes
**ISYE 6740 | Summer 2026 | Group 078**

This notebook performs:
1. **Exploratory Data Analysis** with club-branded visualizations
2. **PCA** for dimensionality reduction
3. **Gaussian Mixture Model (GMM)** soft clustering to identify playstyles
4. **Cluster interpretation** and tactical profiling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from PIL import Image
import json
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add parent dir for club metadata
sys.path.insert(0, os.path.join('..', 'assets'))
from club_metadata import EPL_CLUBS, LA_LIGA_CLUBS, get_all_clubs

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 8),
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
})

CLUB_META = get_all_clubs()
DATA_DIR = os.path.join('..', 'data')
FIG_DIR = os.path.join('..', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print("All imports loaded.")

## 1. Load Data

In [ ]:
# Load cleaned match data from Notebook 1
data_path = os.path.join(DATA_DIR, 'match_data_clean.csv')
feature_path = os.path.join(DATA_DIR, 'feature_columns.json')

if os.path.exists(data_path):
    df = pd.read_csv(data_path, parse_dates=['date'])
    with open(feature_path) as f:
        FEATURE_COLS = json.load(f)
    print(f"Loaded {len(df)} matches with {len(FEATURE_COLS)} features")
else:
    print(">>> DATA NOT FOUND <<<")
    print("Run Notebook 01 first to generate match_data_clean.csv")
    print("\nUsing synthetic placeholder data for development...")
    
    # === PLACEHOLDER: Synthetic data for notebook development ===
    # Replace this block once real data is scraped.
    np.random.seed(42)
    
    FEATURE_COLS = [
        'possession_pct', 'pass_completion_pct', 'progressive_passes',
        'progressive_carries', 'shots', 'shots_on_target', 'xg',
        'pressures', 'successful_pressures', 'pressing_intensity',
        'tackles', 'interceptions', 'aerial_win_rate',
        'crosses_into_box', 'corners', 'carries', 'blocks', 'clearances'
    ]
    
    epl_teams = list(EPL_CLUBS.keys())[:20]
    laliga_teams = list(LA_LIGA_CLUBS.keys())[:20]
    
    rows = []
    for league, teams in [('ENG-Premier League', epl_teams), ('SPA-La Liga', laliga_teams)]:
        for season in ['2023-2024', '2024-2025']:
            for team in teams:
                for match_num in range(38):
                    row = {
                        'date': pd.Timestamp('2023-08-12') + pd.Timedelta(days=match_num*7),
                        'squad': team,
                        'opponent': np.random.choice([t for t in teams if t != team]),
                        'venue': np.random.choice(['Home', 'Away']),
                        'result': np.random.choice(['W', 'D', 'L'], p=[0.4, 0.25, 0.35]),
                        'goals_for': np.random.poisson(1.5),
                        'goals_against': np.random.poisson(1.2),
                        'league': league,
                        'season': season,
                        # Playstyle features with team-specific tendencies
                        'possession_pct': np.clip(np.random.normal(
                            55 if team in ['Manchester City', 'Arsenal', 'Barcelona', 'Real Madrid'] else 48, 8), 25, 80),
                        'pass_completion_pct': np.clip(np.random.normal(
                            85 if team in ['Manchester City', 'Barcelona'] else 78, 5), 60, 95),
                        'progressive_passes': np.random.poisson(45),
                        'progressive_carries': np.random.poisson(30),
                        'shots': np.random.poisson(13),
                        'shots_on_target': np.random.poisson(5),
                        'xg': np.clip(np.random.normal(1.5, 0.7), 0, 5),
                        'pressures': np.random.poisson(
                            180 if team in ['Liverpool', 'Arsenal', 'Atletico Madrid'] else 150),
                        'successful_pressures': np.random.poisson(50),
                        'pressing_intensity': np.clip(np.random.normal(0.30, 0.05), 0.1, 0.5),
                        'tackles': np.random.poisson(18),
                        'interceptions': np.random.poisson(10),
                        'aerial_win_rate': np.clip(np.random.normal(0.50, 0.08), 0.2, 0.8),
                        'crosses_into_box': np.random.poisson(5),
                        'corners': np.random.poisson(5),
                        'carries': np.random.poisson(400),
                        'blocks': np.random.poisson(12),
                        'clearances': np.random.poisson(20),
                    }
                    rows.append(row)
    
    df = pd.DataFrame(rows)
    df['rolling_form_5'] = df.groupby(['league', 'season', 'squad'])['result'].transform(
        lambda x: x.map({'W': 3, 'D': 1, 'L': 0}).rolling(5, min_periods=1).mean()
    )
    print(f"Generated {len(df)} synthetic match observations")
    print("NOTE: Replace with real data from Notebook 01 before final analysis.")

# Quick overview
print(f"\nDataset: {df.shape}")
print(f"Leagues: {df['league'].unique()}")
print(f"Teams per league: {df.groupby('league')['squad'].nunique().to_dict()}")

---
## 2. Exploratory Data Analysis

All visualizations use official club colors and crests where available.

In [ ]:
def get_club_color(team, default='#666666'):
    """Get primary color for a club."""
    if team in CLUB_META:
        return CLUB_META[team]['primary']
    return default

def get_club_colors_list(teams):
    """Get ordered list of colors for a list of teams."""
    return [get_club_color(t) for t in teams]

def load_crest(team, size=0.06):
    """
    Load club crest image for matplotlib annotations.
    >>> PLACEHOLDER: Download crests to assets/crests/{team_short}.png <<<
    Returns None if crest not found.
    """
    if team not in CLUB_META:
        return None
    short = CLUB_META[team].get('short', '')
    crest_path = os.path.join('..', 'assets', 'crests', f'{short}.png')
    if os.path.exists(crest_path):
        img = Image.open(crest_path)
        return OffsetImage(img, zoom=size)
    return None

print("Visualization helpers loaded.")

### 2.1 Feature Distributions by League

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('Key Feature Distributions: EPL vs La Liga', fontsize=20, fontweight='bold', y=1.02)

key_features = [
    ('possession_pct', 'Possession %'),
    ('pass_completion_pct', 'Pass Completion %'),
    ('xg', 'Expected Goals (xG)'),
    ('pressures', 'Pressures'),
    ('pressing_intensity', 'Pressing Success Rate'),
    ('progressive_passes', 'Progressive Passes'),
    ('tackles', 'Tackles'),
    ('aerial_win_rate', 'Aerial Win Rate'),
    ('shots', 'Shots'),
]

# Map to display names for charts
LEAGUE_DISPLAY = {l: 'EPL' if 'Premier' in l or l == 'EPL' else 'La Liga' for l in df['league'].unique()}
df['league_display'] = df['league'].map(LEAGUE_DISPLAY)
league_colors = {'EPL': '#3d195b', 'La Liga': '#ee8707'}

for ax, (feat, title) in zip(axes.flat, key_features):
    if feat in df.columns:
        for league, color in league_colors.items():
            league_data = df[df['league_display'] == league][feat].dropna()
            ax.hist(league_data, bins=30, alpha=0.5, color=color, label=league, density=True)
            ax.axvline(league_data.mean(), color=color, linestyle='--', linewidth=2)
        ax.set_title(title, fontweight='bold')
        ax.legend(frameon=True, fancybox=True)
    else:
        ax.text(0.5, 0.5, f'{feat}\n(not available)', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_distributions_by_league.png'))
plt.show()
print("Saved: figures/feature_distributions_by_league.png")

### 2.2 Team Playstyle Profiles (Radar Charts)

In [ ]:
def plot_team_radar(df, teams, features, title='Team Playstyle Radar'):
    """
    Radar chart comparing average playstyle profiles of selected teams.
    All features are min-max scaled to [0, 1] for comparison.
    """
    # Compute team averages
    team_avgs = df[df['squad'].isin(teams)].groupby('squad')[features].mean()
    
    # Min-max scale across all teams (not just selected)
    all_avgs = df.groupby('squad')[features].mean()
    scaled = (team_avgs - all_avgs.min()) / (all_avgs.max() - all_avgs.min())
    
    # Radar plot
    labels = [f.replace('_', ' ').title() for f in features]
    num_vars = len(features)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    ax.set_title(title, size=18, fontweight='bold', pad=30)
    
    for team in teams:
        if team in scaled.index:
            values = scaled.loc[team].values.flatten().tolist()
            values += values[:1]
            color = get_club_color(team)
            ax.plot(angles, values, 'o-', linewidth=2.5, label=team, color=color)
            ax.fill(angles, values, alpha=0.15, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=10)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12,
              frameon=True, fancybox=True, shadow=True)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig


# Radar features (subset for readability)
radar_features = [
    'possession_pct', 'pass_completion_pct', 'progressive_passes',
    'xg', 'pressures', 'pressing_intensity',
    'tackles', 'interceptions', 'aerial_win_rate', 'crosses_into_box'
]
radar_features = [f for f in radar_features if f in df.columns]

# EPL "Big 6" radar
fig = plot_team_radar(
    df[df['league'].isin(['EPL', 'ENG-Premier League'])],
    ['Manchester City', 'Arsenal', 'Liverpool', 'Chelsea', 'Tottenham', 'Manchester Utd'],
    radar_features,
    title='EPL Big 6: Playstyle Profiles'
)
fig.savefig(os.path.join(FIG_DIR, 'radar_epl_big6.png'))
plt.show()

In [ ]:
# La Liga top teams radar
fig = plot_team_radar(
    df[df['league'].isin(['La Liga', 'SPA-La Liga'])],
    ['Barcelona', 'Real Madrid', 'Atletico Madrid', 'Real Sociedad', 'Athletic Club', 'Villarreal'],
    radar_features,
    title='La Liga Top Teams: Playstyle Profiles'
)
fig.savefig(os.path.join(FIG_DIR, 'radar_laliga_top.png'))
plt.show()

### 2.3 Possession vs Pressing: League-Wide View

In [ ]:
# Average possession vs pressing intensity per team, with club colors
team_avgs = df.groupby(['squad', 'league']).agg({
    'possession_pct': 'mean',
    'pressures': 'mean',
    'xg': 'mean',
}).reset_index()

fig, ax = plt.subplots(figsize=(16, 10))

for _, row in team_avgs.iterrows():
    team = row['squad']
    color = get_club_color(team, '#888888')
    marker = 'o' if 'Premier' in str(row['league']) or row['league'] == 'EPL' else 's'
    edge = '#3d195b' if 'Premier' in str(row['league']) or row['league'] == 'EPL' else '#ee8707'
    
    ax.scatter(
        row['possession_pct'], row['pressures'],
        s=row['xg'] * 200 + 50,  # size = xG
        c=color, edgecolors=edge, linewidth=2,
        marker=marker, alpha=0.85, zorder=5
    )
    
    # Try to load crest, fall back to text label
    short = CLUB_META.get(team, {}).get('short', team[:3].upper())
    ax.annotate(
        short, (row['possession_pct'], row['pressures']),
        textcoords='offset points', xytext=(8, 8),
        fontsize=8, fontweight='bold', color='#333'
    )

# Legend for leagues
epl_patch = mpatches.Patch(edgecolor='#3d195b', facecolor='lightgray', linewidth=2, label='EPL (circles)')
laliga_patch = mpatches.Patch(edgecolor='#ee8707', facecolor='lightgray', linewidth=2, label='La Liga (squares)')
ax.legend(handles=[epl_patch, laliga_patch], loc='upper left', fontsize=12, frameon=True)

ax.set_xlabel('Avg Possession %', fontsize=14)
ax.set_ylabel('Avg Pressures per Match', fontsize=14)
ax.set_title('Possession vs Pressing Intensity\n(bubble size = avg xG)', 
             fontsize=18, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'possession_vs_pressing.png'))
plt.show()
print("Saved: figures/possession_vs_pressing.png")

### 2.4 Correlation Heatmap

In [ ]:
available_features = [f for f in FEATURE_COLS if f in df.columns]
corr = df[available_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Matrix', fontsize=18, fontweight='bold', pad=20)
ax.set_xticklabels([f.replace('_', ' ').title() for f in available_features], rotation=45, ha='right')
ax.set_yticklabels([f.replace('_', ' ').title() for f in available_features], rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'correlation_heatmap.png'))
plt.show()

### 2.5 Match-to-Match Variability

One key motivation: teams don't play the same way every week. This chart shows within-team variance.

In [ ]:
# Coefficient of variation for key features per team
epl_df = df[df['league'].isin(['EPL', 'ENG-Premier League'])]
cv_features = ['possession_pct', 'pressures', 'xg', 'progressive_passes']
cv_features = [f for f in cv_features if f in epl_df.columns]

team_cv = epl_df.groupby('squad')[cv_features].agg(lambda x: x.std() / x.mean()).reset_index()
team_cv = team_cv.sort_values('possession_pct', ascending=False)

fig, axes = plt.subplots(1, len(cv_features), figsize=(5*len(cv_features), 8), sharey=True)
fig.suptitle('Match-to-Match Variability (Coefficient of Variation)\nHigher = more inconsistent', 
             fontsize=16, fontweight='bold', y=1.02)

for ax, feat in zip(axes, cv_features):
    sorted_cv = team_cv.sort_values(feat, ascending=True)
    colors = [get_club_color(t) for t in sorted_cv['squad']]
    ax.barh(range(len(sorted_cv)), sorted_cv[feat], color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(sorted_cv)))
    ax.set_yticklabels(sorted_cv['squad'], fontsize=9)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('CV')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'match_variability.png'))
plt.show()

---
## 3. PCA: Dimensionality Reduction

We standardize features and apply PCA to reduce the feature space while retaining most variance.

In [ ]:
# Standardize features
available_features = [f for f in FEATURE_COLS if f in df.columns]
X = df[available_features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features used: {available_features}")

In [ ]:
# Fit PCA with all components to analyze variance explained
pca_full = PCA()
pca_full.fit(X_scaled)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Scree plot
ax1.bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
        pca_full.explained_variance_ratio_, color='#3d195b', alpha=0.8, label='Individual')
ax1.plot(range(1, len(cumulative_var) + 1), cumulative_var, 'o-', color='#e90052', 
         linewidth=2, markersize=6, label='Cumulative')
ax1.axhline(y=0.90, color='gray', linestyle='--', alpha=0.7, label='90% threshold')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained')
ax1.set_title('PCA Scree Plot', fontweight='bold')
ax1.legend(frameon=True)

# Find number of components for 90% variance
n_components_90 = np.argmax(cumulative_var >= 0.90) + 1
ax1.axvline(x=n_components_90, color='#e90052', linestyle=':', alpha=0.7)
ax1.annotate(f'{n_components_90} PCs → {cumulative_var[n_components_90-1]:.1%} variance',
             xy=(n_components_90, cumulative_var[n_components_90-1]),
             xytext=(n_components_90 + 2, cumulative_var[n_components_90-1] - 0.1),
             arrowprops=dict(arrowstyle='->', color='#e90052'),
             fontsize=11, color='#e90052', fontweight='bold')

# Feature loadings for PC1 and PC2
loadings = pd.DataFrame(
    pca_full.components_[:2].T,
    columns=['PC1', 'PC2'],
    index=available_features
)
loadings_sorted = loadings.reindex(loadings['PC1'].abs().sort_values(ascending=True).index)

colors = ['#3d195b' if v > 0 else '#e90052' for v in loadings_sorted['PC1']]
ax2.barh(range(len(loadings_sorted)), loadings_sorted['PC1'], color=colors, alpha=0.8)
ax2.set_yticks(range(len(loadings_sorted)))
ax2.set_yticklabels([f.replace('_', ' ').title() for f in loadings_sorted.index], fontsize=10)
ax2.set_xlabel('Loading')
ax2.set_title('PC1 Feature Loadings', fontweight='bold')
ax2.axvline(x=0, color='gray', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'pca_analysis.png'))
plt.show()

print(f"\nComponents needed for 90% variance: {n_components_90}")

In [ ]:
# Fit PCA with selected number of components
N_PCA = n_components_90
pca = PCA(n_components=N_PCA, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Reduced from {X_scaled.shape[1]} features to {X_pca.shape[1]} principal components")
print(f"Total variance retained: {pca.explained_variance_ratio_.sum():.1%}")

---
## 4. GMM Soft Clustering

We use a Gaussian Mixture Model for soft (probabilistic) cluster assignments. Each team-match gets a probability vector over K playstyle clusters.

### 4.1 Selecting K (Number of Clusters)

In [ ]:
K_range = range(2, 11)
bic_scores = []
aic_scores = []
silhouette_scores_list = []

for k in K_range:
    gmm = GaussianMixture(n_components=k, covariance_type='full', 
                          random_state=42, n_init=5, max_iter=300)
    labels = gmm.fit_predict(X_pca)
    
    bic_scores.append(gmm.bic(X_pca))
    aic_scores.append(gmm.aic(X_pca))
    
    if k > 1:
        sil = silhouette_score(X_pca, labels)
        silhouette_scores_list.append(sil)
    else:
        silhouette_scores_list.append(0)
    
    print(f"K={k}: BIC={bic_scores[-1]:.0f}, AIC={aic_scores[-1]:.0f}, Silhouette={silhouette_scores_list[-1]:.3f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(list(K_range), bic_scores, 'o-', color='#3d195b', linewidth=2, markersize=8, label='BIC')
ax1.plot(list(K_range), aic_scores, 's-', color='#e90052', linewidth=2, markersize=8, label='AIC')
ax1.set_xlabel('Number of Clusters (K)', fontsize=13)
ax1.set_ylabel('Score', fontsize=13)
ax1.set_title('BIC / AIC vs Number of Clusters', fontweight='bold')
ax1.legend(fontsize=12, frameon=True)
ax1.set_xticks(list(K_range))

# Mark optimal
optimal_k_bic = list(K_range)[np.argmin(bic_scores)]
ax1.axvline(x=optimal_k_bic, color='gray', linestyle='--', alpha=0.5)
ax1.annotate(f'Optimal K={optimal_k_bic}', xy=(optimal_k_bic, min(bic_scores)),
             xytext=(optimal_k_bic + 1, min(bic_scores)),
             arrowprops=dict(arrowstyle='->', color='#3d195b'),
             fontsize=12, fontweight='bold', color='#3d195b')

ax2.plot(list(K_range), silhouette_scores_list, 'D-', color='#00ff85', 
         linewidth=2, markersize=8, markeredgecolor='#333')
ax2.set_xlabel('Number of Clusters (K)', fontsize=13)
ax2.set_ylabel('Silhouette Score', fontsize=13)
ax2.set_title('Silhouette Score vs Number of Clusters', fontweight='bold')
ax2.set_xticks(list(K_range))

optimal_k_sil = list(K_range)[np.argmax(silhouette_scores_list)]
ax2.axvline(x=optimal_k_sil, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cluster_selection.png'))
plt.show()

print(f"\nOptimal K by BIC: {optimal_k_bic}")
print(f"Optimal K by Silhouette: {optimal_k_sil}")

### 4.2 Fit Final GMM

In [ ]:
# Use BIC-optimal K (adjust manually if qualitative analysis suggests otherwise)
K_FINAL = optimal_k_bic
print(f"Fitting GMM with K={K_FINAL} clusters...")

gmm_final = GaussianMixture(
    n_components=K_FINAL, 
    covariance_type='full',
    random_state=42, 
    n_init=10, 
    max_iter=500
)
gmm_final.fit(X_pca)

# Hard assignments (for visualization)
cluster_labels = gmm_final.predict(X_pca)

# Soft assignments (probability vectors — key output for prediction)
cluster_probs = gmm_final.predict_proba(X_pca)

# Add to dataframe
df['cluster'] = cluster_labels
for k in range(K_FINAL):
    df[f'cluster_{k}_prob'] = cluster_probs[:, k]

print(f"Cluster distribution:")
print(df['cluster'].value_counts().sort_index())
print(f"\nSilhouette score: {silhouette_score(X_pca, cluster_labels):.3f}")

### 4.3 Visualize Clusters in PCA Space

In [ ]:
# Cluster colors (distinct palette)
CLUSTER_COLORS = ['#e90052', '#3d195b', '#00ff85', '#04f5ff', '#ff6900', 
                  '#38003c', '#2dba4e', '#ff2882', '#7c2d12', '#1e40af']

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# Left: colored by cluster
ax = axes[0]
for k in range(K_FINAL):
    mask = cluster_labels == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=CLUSTER_COLORS[k], alpha=0.4, s=30, label=f'Cluster {k}')

# Overlay team centroids with club colors
epl_teams = df[df['league'].isin(['EPL', 'ENG-Premier League'])]['squad'].unique()
for team in epl_teams:
    team_mask = (df['squad'] == team).values
    centroid = X_pca[team_mask].mean(axis=0)
    color = get_club_color(team)
    short = CLUB_META.get(team, {}).get('short', team[:3])
    ax.scatter(centroid[0], centroid[1], c=color, s=200, edgecolors='black', 
               linewidth=1.5, zorder=10, marker='o')
    ax.annotate(short, (centroid[0], centroid[1]), textcoords='offset points',
                xytext=(6, 6), fontsize=8, fontweight='bold')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)', fontsize=13)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)', fontsize=13)
ax.set_title('EPL Match Observations in PCA Space\n(colored by cluster, centroids labeled)', fontweight='bold')
ax.legend(loc='best', fontsize=10, frameon=True)

# Right: La Liga
ax = axes[1]
for k in range(K_FINAL):
    mask = cluster_labels == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=CLUSTER_COLORS[k], alpha=0.4, s=30, label=f'Cluster {k}')

laliga_teams = df[df['league'].isin(['La Liga', 'SPA-La Liga'])]['squad'].unique()
for team in laliga_teams:
    team_mask = (df['squad'] == team).values
    centroid = X_pca[team_mask].mean(axis=0)
    color = get_club_color(team)
    short = CLUB_META.get(team, {}).get('short', team[:3])
    ax.scatter(centroid[0], centroid[1], c=color, s=200, edgecolors='black',
               linewidth=1.5, zorder=10, marker='s')
    ax.annotate(short, (centroid[0], centroid[1]), textcoords='offset points',
                xytext=(6, 6), fontsize=8, fontweight='bold')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)', fontsize=13)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)', fontsize=13)
ax.set_title('La Liga Match Observations in PCA Space\n(colored by cluster, centroids labeled)', fontweight='bold')
ax.legend(loc='best', fontsize=10, frameon=True)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'pca_clusters_both_leagues.png'))
plt.show()

### 4.4 Cluster Profiles: What Does Each Playstyle Look Like?

In [ ]:
# Compute cluster centroids in original feature space
cluster_profiles = df.groupby('cluster')[available_features].mean()

# Normalize for heatmap display
profile_normalized = (cluster_profiles - cluster_profiles.min()) / (cluster_profiles.max() - cluster_profiles.min())

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    profile_normalized.T, annot=cluster_profiles.T.round(1), fmt='',
    cmap='YlOrRd', linewidths=0.5, ax=ax,
    xticklabels=[f'Cluster {k}' for k in range(K_FINAL)],
    yticklabels=[f.replace('_', ' ').title() for f in available_features],
    cbar_kws={'label': 'Relative Intensity (0=lowest, 1=highest)'}
)
ax.set_title('Cluster Playstyle Profiles\n(values = raw means, color = relative intensity)', 
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cluster_profiles_heatmap.png'))
plt.show()

In [ ]:
# Name clusters based on their profiles
# (Manually inspect the heatmap above and assign descriptive names)
# This is a placeholder — update after seeing real cluster profiles.

CLUSTER_NAMES = {
    0: 'High-Press Possession',
    1: 'Direct Counter-Attack',
    2: 'Defensive Block',
    3: 'Balanced Midfield Control',
    # Add more if K > 4
}

# Update only for clusters that exist
CLUSTER_NAMES = {k: v for k, v in CLUSTER_NAMES.items() if k < K_FINAL}
# Fill any missing
for k in range(K_FINAL):
    if k not in CLUSTER_NAMES:
        CLUSTER_NAMES[k] = f'Style {k}'

df['cluster_name'] = df['cluster'].map(CLUSTER_NAMES)

print("Cluster assignments:")
for k, name in CLUSTER_NAMES.items():
    count = (df['cluster'] == k).sum()
    print(f"  Cluster {k} ({name}): {count} observations")

### 4.5 Team Cluster Membership Distribution

Since we use soft clustering, each team has a distribution across playstyles. This stacked bar chart shows the average cluster membership per team.

In [ ]:
prob_cols = [f'cluster_{k}_prob' for k in range(K_FINAL)]

for league_name, league_label in [('ENG-Premier League', 'Premier League'), ('SPA-La Liga', 'La Liga')]:
    league_df = df[df['league'] == league_name]
    team_probs = league_df.groupby('squad')[prob_cols].mean()
    
    # Sort by dominant cluster
    team_probs['dominant'] = team_probs.idxmax(axis=1)
    team_probs = team_probs.sort_values(prob_cols[0], ascending=False)
    team_probs = team_probs.drop('dominant', axis=1)
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    bottom = np.zeros(len(team_probs))
    for k in range(K_FINAL):
        col = f'cluster_{k}_prob'
        vals = team_probs[col].values
        ax.barh(range(len(team_probs)), vals, left=bottom, 
                color=CLUSTER_COLORS[k], alpha=0.85,
                label=f'{CLUSTER_NAMES[k]}', edgecolor='white', linewidth=0.5)
        bottom += vals
    
    ax.set_yticks(range(len(team_probs)))
    
    # Color team labels with club colors
    labels = ax.set_yticklabels(team_probs.index, fontsize=10)
    for label in labels:
        team = label.get_text()
        label.set_color(get_club_color(team, '#333'))
        label.set_fontweight('bold')
    
    ax.set_xlabel('Average Cluster Membership Probability', fontsize=13)
    ax.set_title(f'{league_label}: Soft Cluster Membership by Team\n'
                 f'(Each team\'s average playstyle distribution across all matches)',
                 fontsize=16, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10, frameon=True, fancybox=True)
    ax.set_xlim(0, 1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'cluster_membership_{league_name.lower().replace(" ", "_")}.png'))
    plt.show()

### 4.6 Playstyle Evolution Over the Season

How does a team's playstyle shift across the season?

In [ ]:
def plot_playstyle_evolution(df, team, season='2024-2025'):
    """
    Line chart showing how a team's cluster membership evolves match-by-match.
    """
    team_season = df[(df['squad'] == team) & (df['season'] == season)].sort_values('date')
    
    if team_season.empty:
        print(f"No data for {team} in {season}")
        return
    
    fig, ax = plt.subplots(figsize=(16, 5))
    
    x = range(len(team_season))
    for k in range(K_FINAL):
        col = f'cluster_{k}_prob'
        if col in team_season.columns:
            ax.fill_between(x, 0, team_season[col].values, alpha=0.3, 
                           color=CLUSTER_COLORS[k])
            ax.plot(x, team_season[col].values, linewidth=2, 
                    color=CLUSTER_COLORS[k], label=CLUSTER_NAMES[k])
    
    # Mark results
    if 'result' in team_season.columns:
        for i, (_, row) in enumerate(team_season.iterrows()):
            marker = {'W': '^', 'D': 'o', 'L': 'v'}.get(row.get('result', ''), 'o')
            color = {'W': '#00ff85', 'D': '#ffaa00', 'L': '#ff0000'}.get(row.get('result', ''), 'gray')
            ax.scatter(i, 1.05, marker=marker, color=color, s=50, zorder=10, edgecolors='black', linewidth=0.5)
    
    club_color = get_club_color(team)
    ax.set_title(f'{team} — Playstyle Evolution ({season})', 
                 fontsize=16, fontweight='bold', color=club_color)
    ax.set_xlabel('Match Week', fontsize=12)
    ax.set_ylabel('Cluster Membership Probability', fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.legend(loc='upper right', fontsize=9, frameon=True)
    
    plt.tight_layout()
    return fig


# Plot for a few interesting teams
for team in ['Arsenal', 'Liverpool', 'Manchester City', 'Barcelona']:
    if team in df['squad'].values:
        fig = plot_playstyle_evolution(df, team)
        if fig:
            fig.savefig(os.path.join(FIG_DIR, f'evolution_{team.lower().replace(" ", "_")}.png'))
            plt.show()

---
## 5. Export Clustered Data

Save the dataset with cluster assignments and probabilities for Notebook 3 (prediction).

In [ ]:
# Save clustered data
output_path = os.path.join(DATA_DIR, 'match_data_clustered.csv')
df.to_csv(output_path, index=False)

# Save PCA and GMM parameters for reproducibility
import pickle

models = {
    'scaler': scaler,
    'pca': pca,
    'gmm': gmm_final,
    'feature_cols': available_features,
    'n_clusters': K_FINAL,
    'cluster_names': CLUSTER_NAMES,
}

model_path = os.path.join(DATA_DIR, 'clustering_models.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(models, f)

print(f"Saved clustered data to {output_path}")
print(f"Saved models to {model_path}")
print(f"\nReady for Notebook 03: Prediction & Cross-League Analysis")